# Q1 유지 · Q2 녹/Q3 누락 개선 비교 — dev240 자동 검증

**새 L4 런타임 → 기존 Secret GITHUB_TOKEN 허용 → 런타임 / 모두 실행.** Drive 인증 외 박스 그리기·정답 입력·수작업은 없습니다. 기존 dev/기준선/정상 참조/원본 ZIP을 읽고 새 폴더에만 저장합니다.

기본 InternVL, **dev240장 전체**. Q1은 기존 답 그대로, 재추론 0회입니다. 이미 원인 분석에 쓴 40장(`design_dev40`)과 나머지 200장(`additional_dev`)을 별도 채점합니다. 추가200장은 이번 40장 진단에서 제외한 개발 표본이며 정식 최종 holdout은 아닙니다. 개선 판단은 추가200장 표를 먼저 보세요.

| 조건 | Q2 | Q3 |
|---|---|---|
| baseline | 기존 답 | 기존 직접 질문 |
| previous_agent | 이전 자기 답+정의 검토 | 이전 유무→있을 때만 위치 |
| fresh_define | 이전 정의 그대로, 자기 이전 답/재검토 문구 제거 | 직접 질문 |
| q2_cues | 검고 거친 녹·유리 원판 결손·부착 이물 단서, 이전 답 없음 | 직접 질문 |
| q2_cues_q3_position | 수정 Q2 | 모든 사진에서 수정 위치 질문 |
| q2_cues_q3_review | 수정 Q2 | 모든 사진 유무+위치 → 불일치하면 자기 Q3 재검토 |
| matched_reask | 단순 재질문 | 개선 agent와 이미지별 동일 호출 수로 직접 질문 반복 |

Q3는 첫 판단이 '없음'이어도 위치를 확인합니다. 유무와 위치가 어긋나면 두 자기 Q3 답을 이미지로 다시 검토합니다. 어느 한 답을 강제로 따르지 않습니다. 다른 문항 답으로 덮어쓰거나 정답 라벨로 경로를 고르지 않습니다. 녹이면 기존 답을 무조건 유지하는 규칙도 없습니다.

**중요:** 프롬프트/경로 개선 후보를 구현한 상태입니다. 실제 효과는 이 실행으로 확인합니다. Q3 자동 점수는 기존 이미지 라벨 기반 `영역 없음/영역 있음`뿐이며 **실제 위치 정확도는 미검증(`q3_location_accuracy=null`)**입니다. 기존 gold 4보기와 다른 9구역+없음 보조 실험입니다. 항상없음/항상영역 대조군과 결함 놓침 비율도 함께 보세요.

총 신규 호출 **2,400~3,120회/모델** (조건끼리 공유하는 호출은 한 번만 계산). 실제 전체 소요 시간은 아직 측정하지 않았습니다. 20호출마다 저장. 중단되면 추론 셀 재실행으로 성공 호출을 재사용하고 실패 및 새로 필요한 후속 호출을 수행합니다. 오류가 남은 조건은 비교표에서 제외되고 셀이 중단됩니다. 무효 답은 별도 기록하고 자기 유효 위치/직접 답으로 돌아갑니다.

출력: `MyDrive/lvlm_insplad_results/gold_v2_q2_q3_refined_dev_v1/`. 이전 40장 결과, dev 기준선, gold, frozen 설정은 보존합니다. 자동 설정 선택·gold 실행·GitHub 결과 게시 없음. 코드/입력/모델을 바꾸면 새 RUN_FOLDER를 사용하세요. 마지막 셀은 결과 ZIP을 다운로드합니다.


In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
DEV_FOLDER = "lvlm_insplad_qa/gold_v2_dev"
DEV_BASELINES_FOLDER = "lvlm_insplad_results/gold_v2_dev_baselines"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_q2_q3_refined_dev_v1"
MODELS = ["internvl3_5_8b"]  # 필요하면 "qwen3_5_4b" 추가 (호출 2배)
N_IMAGES = 240
SEED = 42
CHECKPOINT_EVERY = 20
UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""


In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
DEV_DIR = DRIVE_ROOT / DEV_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
if not RUN_ROOT.name.startswith("gold_v2_q2_q3_refined_dev"):
    raise ValueError("새 실험 전용 RUN_FOLDER를 사용하세요.")
DEV_REFS_DIR = DEV_DIR / "refs"
STAGE1_DIR = DRIVE_ROOT / DEV_BASELINES_FOLDER / "predictions"

REPO_DIR = Path("/content/lvlm_q2_q3_refined_code")

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
if not REPO_DIR.exists():
    run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
else:
    existing_remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    if existing_remote != f"https://github.com/{REPO}.git":
        raise RuntimeError("실행 폴더의 저장소가 다릅니다. 새 런타임을 사용하세요.")
    # A cached checkout is kept; do not silently change its code during resume.
    if not (REPO_DIR / "scripts/run_gold_v2_q2_q3_refined_dev.py").is_file():
        raise RuntimeError("이 런타임은 이전 코드입니다. 새 런타임을 사용하세요.")
del auth, git_env
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0",
     "sentencepiece==0.2.2","pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")
if torch.cuda.get_device_properties(0).total_memory < 20 * 1024**3:
    raise RuntimeError("VRAM 20GB 이상 L4/A100이 필요합니다.")


In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP); print("UNSUP:", UNSUP_ZIP)

summary = json.loads((DEV_DIR / "build_summary.json").read_text(encoding="utf-8"))
if summary.get("quality_status") != "FROZEN":
    raise RuntimeError(f"dev 세트가 FROZEN이 아닙니다: {summary.get('quality_status')}")
DEV_QA = DEV_DIR / summary["file"]
REFS_CSV = DEV_REFS_DIR / "retrieved_refs.csv"
need = [DEV_QA, REFS_CSV] + [STAGE1_DIR / m / "normal_ref_retrieved_diffdate.jsonl" for m in MODELS]
missing = [str(p) for p in need if not p.is_file()]
if missing:
    raise FileNotFoundError("기존 dev 기준선 입력 없음: " + ", ".join(missing))
print(f"입력 확인 완료 | {N_IMAGES}장, Q1 재호출 0회, Q2/Q3 신규 호출 {N_IMAGES*10*len(MODELS)}~{N_IMAGES*13*len(MODELS)}회")


In [ ]:
# 추론 및 자동 평가. 중단되면 이 셀 재실행 (실패한 호출만 재시도)
RUNNER_PATH = REPO_DIR / "scripts/run_gold_v2_q2_q3_refined_dev.py"
if not RUNNER_PATH.is_file():
    raise FileNotFoundError("merge된 main을 사용하세요. 새 런타임에서 다시 열면 됩니다.")
run([sys.executable, RUNNER_PATH, "--dev-qa", DEV_QA, "--stage1-dir", STAGE1_DIR,
     "--refs-csv", REFS_CSV, "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP,
     "--out-dir", RUN_ROOT, "--models", *MODELS, "--n-images", str(N_IMAGES),
     "--seed", str(SEED), "--checkpoint-every", str(CHECKPOINT_EVERY)])


In [ ]:
# 자동 결과 확인 — 추가 dev를 먼저 비교 (Q3는 영역 유무만)
pd.set_option("display.max_rows", 200)
status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
meta = json.loads((RUN_ROOT / "run_metadata.json").read_text(encoding="utf-8"))
print("실패 호출:", meta["row_errors"], "이번 신규:", meta["called_this_run"], "재사용:", meta["reused_calls"])
display(pd.DataFrame(status))
if meta["row_errors"]:
    raise RuntimeError("실패가 남았습니다. 추론 셀을 재실행하세요. 실패 조건은 비교에서 제외됩니다.")
answers = pd.read_csv(RUN_ROOT / "answers.csv")
q1 = answers[answers.question_type == "anomaly_detection"]
assert not q1.empty and (q1.predicted_choice == q1.cached_baseline_choice).all(), "Q1 답 변경 또는 누락"
print("Q1 기존 답 유지:", len(q1), "행 모두 동일, 재추론0회")
metrics = pd.read_csv(RUN_ROOT / "cohort_metrics.csv")
columns = ["model_key", "policy", "images", "q2_defect_macro_f1", "q2_accuracy", "joint_accuracy",
           "q3_presence_balanced_accuracy", "q3_normal_false_region_rate", "q3_anomaly_false_none_rate",
           "q3_invalid_rate", "q1_q2_q3_presence_joint"]
for cohort, label in (("additional_dev", "추가 dev — 먼저 확인"), ("design_dev40", "진단에 사용한 40장"), ("all_dev", "전체 dev")):
    print(label)
    mine = metrics[metrics.cohort == cohort]
    if mine.empty:
        print("표본 없음 (N_IMAGES 설정 확인)")
    else:
        display(mine[columns].round(3))
print("Q3 위치 정확도:", meta["q3_location_accuracy"], "— 미검증, 표의 joint도 Q3 유무만 포함")
print("추가 dev 결함 종류별 Q2 (특히 rust)와 정상/결함 지원 수")
defects = pd.read_csv(RUN_ROOT / "cohort_q2_by_defect.csv")
display(defects[defects.cohort == "additional_dev"].round(3))
assets = pd.read_csv(RUN_ROOT / "cohort_metrics_by_asset.csv")
display(assets[(assets.cohort == "additional_dev") & assets.policy.isin(["baseline", "previous_agent", "q2_cues_q3_review"])][
    ["policy", "asset_category", "q2_accuracy", "q3_presence_balanced_accuracy", "q3_normal_images", "q3_anomalous_images",
     "q3_normal_false_region_rate", "q3_anomaly_false_none_rate"]].round(3))
print("추가 dev paired bootstrap CI: baseline/이전agent/동일호출 대조 및 단계별 ablation")
sig = pd.read_csv(RUN_ROOT / "cohort_significance.csv")
important = ["q2_defect_macro_f1", "joint_accuracy", "q3_presence_balanced_accuracy", "q3_normal_false_region_rate", "q3_anomaly_false_none_rate"]
display(sig[(sig.cohort == "additional_dev") & sig.metric.isin(important)].round(3))
print("전체 dev 항상없음/항상영역 대조군")
display(pd.read_csv(RUN_ROOT / "q3_constant_baselines.csv").round(3))
print("조건별 호출 수와 전체 공유 호출은 run_status / run_metadata에 저장됐습니다.")
print("결과 위치:", RUN_ROOT)


In [ ]:
# 결과 ZIP 다운로드 (Drive 결과는 그대로 유지)
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path("/content/dev_q2_q3_refined_results.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for p in RUN_ROOT.rglob("*"):
        if p.is_file() and "_images" not in p.relative_to(RUN_ROOT).parts:
            z.write(p, p.relative_to(RUN_ROOT))
files.download(str(archive))
